In [1]:
import warnings
import pandas as pd
import numpy as np
from sdv.single_table import (
    GaussianCopulaSynthesizer, TVAESynthesizer, CTGANSynthesizer)
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality

warnings.filterwarnings('ignore')

base_dataset = pd.read_csv('base-datasets/ECU_IoHT.csv')
supporting_dataset = pd.read_csv(
    'base-datasets/wustl-ehms-2020_with_attacks_categories.csv')

In [ ]:
attack_types = ['ARP Spoofing', 'Nmap Port Scan', 'Smurf Attack', 'DoS Attack']
base_dataset['Label'] = base_dataset['Type of attack'].apply(
    lambda x: 1 if x in attack_types else 0)
base_dataset['Type'] = base_dataset['Type of attack'].apply(
    lambda x: x.lower() if x in attack_types else "normal")
base_dataset['Duration'] = base_dataset['Time'].diff().fillna(0)
base_dataset = base_dataset[['Source', 'Destination', 'Protocol',
                             'Duration', 'Length', 'Type', 'Label']].copy()
additional_features = supporting_dataset[['SrcBytes', 'DstBytes', 'SrcLoad',
                                          'DstLoad', 'SIntPkt', 'DIntPkt', 'SrcJitter', 'DstJitter']].copy()

combined_dataset = pd.concat([base_dataset.reset_index(drop=True),
                              additional_features], axis=1)

cols_to_fill = ['SrcBytes', 'DstBytes', 'SrcLoad',
                'DstLoad', 'SIntPkt', 'DIntPkt', 'SrcJitter', 'DstJitter']

for col in cols_to_fill:
    mask = combined_dataset[col].isna()
    if mask.any():
        candidates = combined_dataset.loc[~combined_dataset[col].isna(
        ), col].values
        combined_dataset.loc[mask, col] = np.random.choice(
            candidates, size=mask.sum(), replace=True)
combined_dataset = combined_dataset.drop_duplicates(ignore_index=True)
# combined_dataset.to_csv("base-datasets/synthetic_data.csv", index=False)
combined_dataset.to_csv("synthetic_data.csv", index=False)

In [3]:
def preprocess_data(df):
    """Proper preprocessing that identifies both categorical and numerical columns"""
    df_processed = df.copy()
    preprocessing_info = {}

    # Store original dtypes BEFORE any modifications
    preprocessing_info['original_dtypes'] = df.dtypes.to_dict()
    preprocessing_info['original_columns'] = df.columns.tolist()

    # Identify categorical columns
    categorical_cols = df.select_dtypes(
        include=['object', 'category']).columns.tolist()

    # If Label exists and is numerical, convert it to categorical for CTGAN
    if 'Label' in df.columns and df['Label'].dtype in [np.int64, np.int32, np.float64, np.float32]:
        df_processed['Label'] = df_processed['Label'].astype(str)
        if 'Label' not in categorical_cols:
            categorical_cols.append('Label')

    # Convert all categorical columns to string
    for col in categorical_cols:
        if col in df_processed.columns:
            df_processed[col] = df_processed[col].astype(str)

    # Identify numerical columns (EXCLUDING Label if we converted it)
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Label' in numerical_cols and 'Label' in categorical_cols:
        numerical_cols.remove('Label')

    # Store preprocessing information
    preprocessing_info['categorical_cols'] = categorical_cols
    preprocessing_info['numerical_cols'] = numerical_cols

    return df_processed, preprocessing_info


def postprocess_data(synthetic_df, preprocessing_info):
    synthetic_processed = synthetic_df.copy()
    original_dtypes = preprocessing_info['original_dtypes']

    for col in synthetic_processed.columns:
        if col in original_dtypes:
            original_dtype = original_dtypes[col]

            try:
                # Handle Label column specially
                if col == 'Label':
                    if original_dtype in [np.int64, np.int32, np.int16, np.int8]:
                        # Convert string labels to integers
                        synthetic_processed[col] = pd.to_numeric(
                            synthetic_processed[col], errors='coerce')
                        synthetic_processed[col] = synthetic_processed[col].fillna(
                            0).astype(int)
                        synthetic_processed[col] = synthetic_processed[col].clip(
                            0, 1)

                # Handle other numerical columns
                elif original_dtype in [np.int64, np.int32, np.int16, np.int8, np.float64, np.float32]:
                    synthetic_processed[col] = pd.to_numeric(
                        synthetic_processed[col], errors='coerce')
                    synthetic_processed[col] = synthetic_processed[col].astype(
                        original_dtype)

                # Handle categorical/string columns
                elif original_dtype in ['object', 'category']:
                    synthetic_processed[col] = synthetic_processed[col].astype(
                        str)

            except Exception as e:
                print(
                    f"Warning: Could not convert column {col} to {original_dtype}: {e}")
                # Keep as is if conversion fails

    # Additional cleanup for specific columns
    if "Duration" in synthetic_processed.columns:
        synthetic_processed["Duration"] = pd.to_numeric(
            synthetic_processed["Duration"], errors='coerce')
        synthetic_processed["Duration"] = synthetic_processed["Duration"].fillna(
            0)
        synthetic_processed["Duration"] = synthetic_processed["Duration"].clip(
            lower=0)

    return synthetic_processed


def enforce_label_consistency(synthetic_df):
    """Ensure Type normal = Label 0, other Types = Label 1"""
    if 'Type' in synthetic_df.columns and 'Label' in synthetic_df.columns:
        print("\nEnforcing label consistency based on Type...")

        # Count before correction
        print(f"Before correction:")
        print(
            f"Type normal with Label 0: {len(synthetic_df[(synthetic_df['Type'] == 'normal') & (synthetic_df['Label'] == 0)])}")
        print(
            f"Type normal with Label 1: {len(synthetic_df[(synthetic_df['Type'] == 'normal') & (synthetic_df['Label'] == 1)])}")
        print(
            f"Other Types with Label 0: {len(synthetic_df[(synthetic_df['Type'] != 'normal') & (synthetic_df['Label'] == 0)])}")
        print(
            f"Other Types with Label 1: {len(synthetic_df[(synthetic_df['Type'] != 'normal') & (synthetic_df['Label'] == 1)])}")

        # Apply the rule: Type normal = Label 0, other Types = Label 1
        synthetic_df_corrected = synthetic_df.drop(
            synthetic_df[
                ((synthetic_df['Type'] == 'normal') & (synthetic_df['Label'] == 1)) |
                ((synthetic_df['Type'] != 'normal')
                 & (synthetic_df['Label'] == 0))
            ].index
        )

        # Count after correction
        print(f"\nAfter correction:")
        print(
            f"Type normal with Label 0: {len(synthetic_df_corrected[(synthetic_df_corrected['Type'] == 'normal') & (synthetic_df_corrected['Label'] == 0)])}")
        print(
            f"Type normal with Label 1: {len(synthetic_df_corrected[(synthetic_df_corrected['Type'] == 'normal') & (synthetic_df_corrected['Label'] == 1)])}")
        print(
            f"Other Types with Label 0: {len(synthetic_df_corrected[(synthetic_df_corrected['Type'] != 'normal') & (synthetic_df_corrected['Label'] == 0)])}")
        print(
            f"Other Types with Label 1: {len(synthetic_df_corrected[(synthetic_df_corrected['Type'] != 'normal') & (synthetic_df_corrected['Label'] == 1)])}")

    return synthetic_df_corrected

In [4]:
processed_df, preprocessing_info = preprocess_data(combined_dataset)

# Initialize metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_df)

# Training configuration
epochs = 1000
batch_size = 1000
num_samples = len(processed_df)

# Define augmentation techniques with consistent parameters
aug_techniques = {
    'TVAE': {
        'class': TVAESynthesizer,
        'params': {
            'metadata': metadata,
            'epochs': epochs,
            'verbose': True
        }
    },
    'Gaussian_Copula': {
        'class': GaussianCopulaSynthesizer,
        'params': {
            'metadata': metadata
        }
    },
    'CT_GAN': {
        'class': CTGANSynthesizer,
        'params': {
            'metadata': metadata,
            'epochs': epochs,
            'batch_size': batch_size,
            'verbose': True,
            'cuda': True,
            'generator_dim': (256, 256, 256),
            'discriminator_dim': (256, 256, 256),
            'generator_lr': 2e-4,
            'discriminator_lr': 2e-4,
            'generator_decay': 1e-6,
            'discriminator_decay': 1e-6,
            'discriminator_steps': 1,
            'pac': 10,
            'log_frequency': True
        }
    },
}

for name, technique_config in aug_techniques.items():
    print(f"Training {name} synthesizer...")

    # Initialize model
    model = technique_config['class'](**technique_config['params'])
    model.fit(processed_df)

    synthetic_data = model.sample(num_samples)

    synthetic_data = postprocess_data(synthetic_data, preprocessing_info)
    synthetic_data = enforce_label_consistency(synthetic_data)
    synthetic_data = synthetic_data.drop_duplicates(ignore_index=True)

    quality_report = evaluate_quality(
        combined_dataset, synthetic_data, metadata)

    # synthetic_data.to_csv(f"base-datasets/synthetic_data_{name.lower()}.csv", index=False)
    synthetic_data.to_csv(f"synthetic_data_{name.lower()}.csv", index=False)

    print(f"Completed {name} - Generated {len(synthetic_data)} samples")

Training TVAE synthesizer...


Loss: -56.990: 100%|██████████| 1000/1000 [1:26:16<00:00,  5.18s/it] 



Enforcing label consistency based on Type...
Before correction:
Type normal with Label 0: 20855
Type normal with Label 1: 736
Other Types with Label 0: 336
Other Types with Label 1: 89280

After correction:
Type normal with Label 0: 20855
Type normal with Label 1: 0
Other Types with Label 0: 0
Other Types with Label 1: 89280
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 17.45it/s]|
Column Shapes Score: 94.07%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:02<00:00, 40.42it/s]|
Column Pair Trends Score: 74.15%

Overall Score (Average): 84.11%

Completed TVAE - Generated 110135 samples
Training Gaussian_Copula synthesizer...

Enforcing label consistency based on Type...
Before correction:
Type normal with Label 0: 8809
Type normal with Label 1: 14367
Other Types with Label 0: 14443
Other Types with Label 1: 73588

After correction:
Type normal with Label 0: 8809
Type normal with Label 1: 0
Other Types with Label 0: 0
Other Types 

Gen. (-0.29) | Discrim. (-0.25): 100%|██████████| 1000/1000 [3:13:10<00:00, 11.59s/it] 



Enforcing label consistency based on Type...
Before correction:
Type normal with Label 0: 62464
Type normal with Label 1: 2730
Other Types with Label 0: 804
Other Types with Label 1: 45209

After correction:
Type normal with Label 0: 62464
Type normal with Label 1: 0
Other Types with Label 0: 0
Other Types with Label 1: 45209
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 35.21it/s]|
Column Shapes Score: 71.17%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:01<00:00, 86.93it/s]| 
Column Pair Trends Score: 54.7%

Overall Score (Average): 62.94%

Completed CT_GAN - Generated 107673 samples
